# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [35]:
import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

## Parameters

In [36]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 2, 1)
END_DATE   = datetime.date(2026, 3, 11)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-5% wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 1)

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../data/trades_polygon")
MARKETS_DIR = Path("../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [37]:
market_files = sorted(MARKETS_DIR.glob("*.parquet"))
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 66 market file(s)
Unique markets (all):  577,689
Unique markets (filtered 2025-02-01 → 2026-03-11): 485,346


,end_date_iso,condition_id,market_json
0,2025-02-01T00:00:00Z,0x04dbe85cf31a4198d8bd414aa18d7868ab9987701a035536ab7e151aeb31970a,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-01-21T05:17:10Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x04dbe85cf31a4198d8bd414aa18d7868ab9987701a035536ab7e151aeb31970a"",""question_id"":""0xc0479cbfb05b2aa9360f28dd4295bb98b1140f62c52864d126068d4551033a00"",""question"":""Will Newcastle win on 2025-02-01?"",""description"":""In the upcoming EPL game, scheduled for February 1 at 10:00AM ET,\nIf Newcastle wins, this market will resolve to “Yes”.\nIf Newcastle loses, this market will resolve to “No”.\nIf the game is postponed, this market will remain open until the game has been completed.\nIf the game is canceled entirely, with no make-up game, this market will resolve “No”."",""market_slug"":""epl-new-ful-2025-02-01-new"",""end_date_iso"":""2025-02-01T00:00:00Z"",""game_start_time"":""2025-02-01T15:00:00Z"",""seconds_delay"":3,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":false,""neg_risk"":true,""neg_risk_market_id"":""0xc0479cbfb05b2aa9360f28dd4295bb98b1140f62c52864d126068d4551033a00"",""neg_risk_request_id"":""0x4462fa4ce9e81c70406e488b8f5067280ada4532205f143eeaab45f902aeee94"",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/epl_newcastle.png"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/epl_newcastle.png"",""rewards"":{""rates"":null,""min_size"":0,""max_spread"":0},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""29141486809745035296786432305120857548031802523257562069336273990325051755559"",""outcome"":""Yes"",""price"":0,""winner"":false},{""token_id"":""100120619122482916369663717268212818789615681029830955683568535217562904154968"",""outcome"":""No"",""price"":1,""winner"":true}],""tags"":[""Sports"",""Premier League"",""EPL"",""Games""]}"
1,2025-02-01T00:00:00Z,0x05abfe3cb89504134856c42c701bbe77b3f26fa3d4dfcbd7a3d8e76b143fd288,"{""enable_order_book"":false,""active"":true,""closed"":true,""archived"":false,""accepting_orders"":false,""accepting_order_timestamp"":""2025-01-18T19:51:22Z"",""minimum_order_size"":5,""minimum_tick_size"":0.001,""condition_id"":""0x05abfe3cb89504134856c42c701bbe77b3f26fa3d4dfcbd7a3d8e76b143fd288"",""question_id"":""0xe0cb250f1ba956286ed64c422b501fbe551e78914568f63ab1f3e10b8e11d800"",""question"":""Will $Trump FDV be less than $5b on Feb 1?"",""description"":""This market will resolve to \""Yes\"" if the $TRUMP 1 minute candle for February 1, 12:00 in the ET timezone (noon) has a final “Close” FDV of $5,000,000,000.00 or less. Otherwise, this market will resolve to \""No\"".\n\nThe primary resolution source for this market will be 1 minute price candles for $TRUMP available at https://dexscreener.com/solana/a8nphpcjqtqhdquk35uj9hy2ysgxfkczgunwvkd3k7vc, viewed by clicking Price and 1m on the top row of the chart, when multiplied by the total supply."",""market_slug"":""will-trump-fdv-be-less-than-5b-on-feb-1"",""end_date_iso"":""2025-02-01T00:00:00Z"",""game_start_time"":null,""seconds_delay"":0,""fpmm"":"""",""maker_base_fee"":0,""taker_base_fee"":0,""notifications_enabled"":true,""neg_risk"":true,""neg_risk_market_id"":""0xe0cb250f1ba956286ed64c422b501fbe551e78914568f63ab1f3e10b8e11d800"",""neg_risk_request_id"":""0xc7a69e078b20b6b9778ca4f5e79a1a2917b544a3d8c1374e6e31d27d9ded0e05"",""icon"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/trump-fdv-on-february-1-1YFcStkLYqrE.jpg"",""image"":""https://polymarket-upload.s3.us-east-2.amazonaws.com/trump-fdv-on-february-1-1YFcStkLYqrE.jpg"",""rewards"":{""rates"":[{""asset_address"":""0x2791Bca1f2de4661ED88A30C99A7a9449Aa84174"",""rewards_daily_rate"":10}],""min_size"":50,""max_spread"":3.5},""is_50_50_outcome"":false,""tokens"":[{""token_id"":""75410493532680363944213231557707838041093316509840066207111

In [38]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Token lookup entries: 967,309
Market meta rows:     485,346


## 2 · Process partitioned per-side trades and filter to in-range markets

Trades are already stored as wallet-perspective rows (`wallet`, `side`) and
`TRADES_DIR` is partitioned, so we process each parquet shard independently to
avoid loading all fills into memory at once. For each shard we:
1. Inner-join with token resolution (`token_winner`, `final_price`)
2. Build `final_value_usdc = quantity × final_price`
3. Aggregate to wallet × market × timestamp and accumulate global results


In [39]:
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade parquet file(s)")

sample_raw = pd.read_parquet(trade_files[0])
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Sample shard date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)


Found 16 trade parquet file(s)
Sample shard rows (0.parquet): 5,498,880
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Sample shard date range: 2025-01-01 → 2026-03-11


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,wallet,side,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,78753205165658130534456351077321496563862891920229742523513427553266682271361,No,0x126f15c7bd21bf5bf136f3629e10990dc0677137,BUY,...,1,10.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
1,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e3ddf9c6c7100eacb34fc,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,BUY,...,1,10.0,True,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",SELL,No,78753205165658130534456351077321496563862891920229742523513427553266682271361,0.62,6.2
2,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38fce30001acc0181bf112,1531,66159089,1735689997,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcdf60247b994f132ab79bb,58665055277986534416719939405914621458010137331379938342097742389618466217100,Yes,0x260d1ae03c985f7c78ad286b5da14940b4739679,BUY,...,1,30.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 2025 Predictions]",BUY,Yes,58665055277986534416719939405914621458010137331379938342097742389618466217100,0.37,11.1


In [40]:
# ── build token resolution DataFrame from lookup ─────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

def enrich_trade_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Attach settlement columns and computed datetime/value to one shard."""
    if chunk.empty:
        return chunk

    c = chunk.copy()
    c["token_id"] = c["token_id"].astype(str)
    c = c.merge(
        token_df[["token_id", "token_winner", "final_price"]],
        on="token_id",
        how="inner",
    )
    if c.empty:
        return c

    c["dt"] = pd.to_datetime(c["block_timestamp"], unit="s", utc=True)
    c["final_value_usdc"] = c["quantity"] * c["final_price"]
    return c

GROUP_KEYS = ["wallet", "condition_id", "dt"]

READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "trade_date", "condition_id",
    "token_id", "outcome", "price", "quantity", "usdc_amount", "position",
    "wallet", "side",
]

grouped_parts: list[pd.DataFrame] = []
sample_fills = None

total_raw_rows = 0
total_filtered_rows = 0

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=READ_COLS)
    total_raw_rows += len(raw_chunk)

    enriched_chunk = enrich_trade_chunk(raw_chunk)
    total_filtered_rows += len(enriched_chunk)

    if sample_fills is None and len(enriched_chunk) > 0:
        sample_fills = enriched_chunk.head(5).copy()

    if len(enriched_chunk) == 0:
        continue

    grouped_part = (
        enriched_chunk.groupby(GROUP_KEYS, sort=False)
        .agg(
            side             = ("side",             "first"),
            outcome          = ("outcome",          "first"),
            token_winner     = ("token_winner",     "first"),
            final_price      = ("final_price",      "first"),
            position         = ("position",         "max"),
            total_quantity   = ("quantity",         "sum"),
            price_sum        = ("price",            "sum"),
            price_count      = ("price",            "count"),
            trade_value_usdc = ("usdc_amount",      "sum"),
            final_value_usdc = ("final_value_usdc", "sum"),
            num_fills        = ("tx_hash",          "count"),
        )
        .reset_index()
    )
    grouped_parts.append(grouped_part)

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed {i}/{len(trade_files)} shards | "
            f"raw rows: {total_raw_rows:,} | in-range rows: {total_filtered_rows:,}"
        )

if not grouped_parts:
    raise ValueError("No in-range trade rows after token filter")

grouped = pd.concat(grouped_parts, ignore_index=True)

# In case a key spans multiple shards, combine partial aggregates.
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        side             = ("side",             "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_sum        = ("price_sum",        "sum"),
        price_count      = ("price_count",      "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_sum"] / grouped["price_count"]
grouped.drop(columns=["price_sum", "price_count"], inplace=True)
grouped.sort_values(["wallet", "condition_id", "dt"], inplace=True, ignore_index=True)

print(
    f"Rows after market filter: {total_filtered_rows:,}  "
    f"(dropped {total_raw_rows - total_filtered_rows:,} rows outside [{START_DATE}, {END_DATE}])"
)
print(f"Prepared grouped rows: {len(grouped):,}")
if sample_fills is not None:
    sample_fills.head(5)


Processed 16/16 shards | raw rows: 93,357,019 | in-range rows: 93,357,019
Rows after market filter: 93,357,019  (dropped 0 rows outside [2025-02-01, 2026-03-11])
Prepared grouped rows: 66,671,178


## 3 · Enriched fill table

All fills are already filtered to in-range markets and carry resolution columns
from the inner join performed in the previous step.

| column | description |
|---|---|
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Settlement price: 1.0 if `token_winner=True`, 0.0 otherwise |
| `final_value_usdc` | `quantity × final_price` — USDC value of the position at settlement |

In [41]:
if sample_fills is None:
    print("No enriched fills available after filtering")
    pd.DataFrame(columns=[
        "wallet", "side", "condition_id", "dt", "outcome", "quantity", "price",
        "position", "usdc_amount", "token_winner", "final_price", "final_value_usdc"
    ])
else:
    print(f"Raw rows scanned across all shards: {total_raw_rows:,}")
    print(f"In-range rows after token join:     {total_filtered_rows:,}")
    print(f"token_winner null: {sample_fills['token_winner'].isna().sum():,}  (sample should be 0)")
    sample_fills[["wallet", "side", "condition_id", "dt", "outcome", "quantity", "price", "position",
                  "usdc_amount", "token_winner", "final_price", "final_value_usdc"]].head(5)


Raw rows scanned across all shards: 93,357,019
In-range rows after token join:     93,357,019
token_winner null: 0  (sample should be 0)


## 4 · Group by wallet × market × timestamp

A single on-chain transaction can generate multiple fills at the same timestamp.
Each row in `grouped` aggregates all fills for one wallet in one market at one timestamp.

In [42]:
print(f"Grouped rows: {len(grouped):,}")
grouped.head(10)


Grouped rows: 66,671,178


,wallet,condition_id,dt,side,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,avg_price
0,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 06:27:41+00:00,BUY,No,False,0.0,10435.000000,10435.000000,20.870000,0.000000,2,0.002
1,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 07:25:19+00:00,BUY,No,False,0.0,20870.000000,10435.000000,20.870000,0.000000,2,0.002
2,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 07:26:03+00:00,SELL,No,False,0.0,18007.010000,2862.990000,5.725980,0.000000,1,0.002
3,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 07:26:53+00:00,SELL,No,False,0.0,10435.000000,7572.010000,15.144020,0.000000,1,0.002
4,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 09:11:33+00:00,BUY,No,False,0.0,20650.000000,10215.000000,20.430000,0.000000,2,0.002
5,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 09:12:17+00:00,SELL,No,False,0.0,17206.120000,3443.880000,6.887760,0.000000,1,0.002
6,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 09:12:53+00:00,SELL,No,False,0.0,10435.000000,6771.120000,13.542240,0.000000,1,0.002
7,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x21d5b7820786486cce744024ca423015bf5c2b03530620ef4436a5f8efb9dbe7,2025-12-30 07:02:51+00:00,BUY,No,True,1.0,1.681000,1.681000,1.679319,1.681000,1,0.999
8,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0xdc840e634d22dffd154de05dd4efe3cab4e8c3f43e9f6b806dd0699747a9bbda,2025-11-11 11:28:13+00:00,BUY,Down,True,1.0,2.439023,2.439023,0.999999,2.439023,1,0.410
9,0x0000000433698c8f7f08eb24c57f1d62a6fd946e,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c56944c6ad5c945e9a2ba7211,2026-02-14 01:47:41+00:00,BUY,No,True,1.0,1.184833,1.184833,0.999999,1.184833,1,0.844


## 5 · Per-trade P&L and wallet summary (training data only)

P&L is calculated **per trade** based on whether the traded token is a winner:

| side | `trade_pnl` formula | intuition |
|---|---|---|
| BUY | `final_value_usdc − trade_value_usdc` | bought tokens; profit if `final_price > entry_price` |
| SELL | `trade_value_usdc − final_value_usdc` | sold tokens; profit if sold above final settlement price |

where `final_value_usdc = total_quantity × final_price` (1.0 if the token is a winner, 0.0 otherwise).

Wallet P&L = `Σ trade_pnl` over training trades.

In [43]:
# ── per-trade P&L: final_value vs. trade_value, signed by side ──────────────
# BUY:  wallet paid trade_value_usdc, ends up with tokens worth final_value_usdc
#       → pnl = final_value_usdc - trade_value_usdc  (positive if token wins)
# SELL: wallet received trade_value_usdc, gave up tokens worth final_value_usdc
#       → pnl = trade_value_usdc - final_value_usdc  (positive if token loses)
is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = (
    (grouped["final_value_usdc"] - grouped["trade_value_usdc"]).where(is_buy,
    grouped["trade_value_usdc"] - grouped["final_value_usdc"])
)

# ── restrict to training period for wallet ranking ────────────────────────────
end_train_ts   = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train  = grouped[grouped["dt"] < end_train_ts]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets      = ("condition_id",   "nunique"),
        num_trades       = ("num_fills",       "sum"),
        total_cost_usdc  = ("trade_value_usdc", "sum"),
        pnl_usdc         = ("trade_pnl",       "sum")
                )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.head(5)

Unique wallets (training period): 908,136


,wallet,num_markets,num_trades,total_cost_usdc,pnl_usdc
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,1746,108053,9.085269e+07,7.357349e+06
1,0x16b29c50f2439faf627209b2ac0c7bbddaa8a881,5507,95122,6.967748e+07,3.215593e+06
2,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,521,49115,3.107744e+07,3.170055e+06
3,0xdc876e6873772d38716fda7f2452a78d426d7ab6,1924,64564,3.511902e+07,2.622418e+06
4,0xd38b71f3e8ed1af71983e5c309eac3dfa9b35029,437,6866,1.036048e+07,2.258163e+06


## 6 · Market-level volume summary

In [44]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum")
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 214,148


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5,US government shutdown Saturday?,2026-01-31 00:00:00+00:00,27577,351968,1.394019e+08,1.416030e+08
1,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,Will Zelenskyy wear a suit before July?,2025-06-30 00:00:00+00:00,7435,101246,1.365801e+08,1.356567e+08
2,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Khamenei out as Supreme Leader of Iran by February 28?,2026-02-28 00:00:00+00:00,22553,280093,1.089277e+08,1.182862e+08
3,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,"US strikes Iran by February 28, 2026?",2026-01-31 00:00:00+00:00,22906,268526,7.830458e+07,9.590727e+07
4,0xef8cf8b45ef7e3a607a72b6e1d56bede869fdd81795b63db847de1090bf11c41,TikTok banned in the US before May 2025?,2025-04-30 00:00:00+00:00,7851,64858,7.655462e+07,7.642088e+07
5,0xf2ce8d3897ac5009a131637d3575f1f91c579bd08eecce6ae2b2da0f32bbe6f1,Xi Jinping out in 2025?,2025-12-31 00:00:00+00:00,27608,253840,6.380369e+07,6.416083e+07
6,0x32707ae194389a70f25314ec934851daba12d893da616c492ffe8b655b7759a9,"Will Eleven die in ""Stranger Things: Season 5""?",2025-12-31 00:00:00+00:00,5989,55012,6.210652e+07,6.187640e+07
7,0x77b4a1d4cd398a86c18b6bb9b5218917dc9f04b01a3cd4bae1c71ea345f15fa8,Will Polymarket US go live in 2025?,2025-12-31 00:00:00+00:00,18072,153627,5.097443e+07,5.120541e+07
8,0x8ee2f1640386310eb5e7ffa596ba9335f2d324e303d21b0dfea6998874445791,Russia x Ukraine ceasefire in 2025?,2025-12-31 00:00:00+00:00,31318,220730,5.066134e+07,4.778616e+07
9,0x62b0cd598091a179147acbd4616400f804acfdff6f76f029944b481b37cbd45f,US x Venezuela military engagement by December 31?,2025-12-31 00:00:00+00:00,10771,143113,4.697017e+07,4.605230e+07


## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets (training period).

In [53]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc,wallet_count_complement
quantile,,,,,
0.001,908,1.0,1.0,-32407.39,907228
0.01,9081,1.0,1.0,-3270.84,899055
0.05,45407,1.0,1.0,-456.56,862729
0.1,90814,1.0,1.0,-121.51,817322
0.25,227034,2.0,1.0,-13.96,681102
0.5,454068,5.0,3.0,-0.18,454068
0.75,681102,15.0,7.0,2.57,227034
0.9,817322,41.0,17.0,45.17,90814
0.95,862729,94.0,36.0,226.79,45407


## 8 · Full enriched trade table

One row per wallet × market × timestamp, with all enrichment columns.

In [46]:
DISPLAY_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,num_fills
0,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 06:27:41+00:00,BUY,No,10435.00,10435.00,0.002,False,0.0,20.87000,0.0,-20.87000,2
1,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 07:25:19+00:00,BUY,No,20870.00,10435.00,0.002,False,0.0,20.87000,0.0,-20.87000,2
2,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 07:26:03+00:00,SELL,No,18007.01,2862.99,0.002,False,0.0,5.72598,0.0,5.72598,1
3,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 07:26:53+00:00,SELL,No,10435.00,7572.01,0.002,False,0.0,15.14402,0.0,15.14402,1
4,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,2025-11-04 09:11:33+00:00,BUY,No,20650.00,10215.00,0.002,False,0.0,20.43000,0.0,-20.43000,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66671173,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,2026-02-05 16:37:39+00:00,BUY,Yes,150.00,150.00,0.080,True,1.0,12.00000,150.0,138.00000,3
66671174,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,2026-02-05 17:13:43+00:00,BUY,Yes,350.00,200.00,0.080,True,1.0,16.00000,200.0,184.00000,1
66671175,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,2026-02-05 22:23:27+00:00,BUY,Yes,550.00,200.00,0.070,True,1.0,14.00000,200.0,186.00000,1
66671176,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,2026-02-28 16:39:01+00:00,SELL,Yes,530.00,50.00,0.507,True,1.0,25.35000,50.0,-24.65000,2


## 9 · Export top-5% wallets to parquet shards

Identifies top-5% wallets by P&L on the **training period**, then performs a
second partition-by-partition pass to export only those wallets' trades from the
**full dataset** (training + test) to `data/polygon_trades_processed/*.parquet`.

Each row is a single per-side fill, enriched with `final_value_usdc` and a boolean
`is_train` flag. Output remains partitioned by input shard.


In [47]:
# ── identify top-5% wallets by training-period P&L ───────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:                {len(top_wallets):,}")

# print deciles of training P&L for top wallets
top_pnl = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"]
for i in range(0, 11):
    q = i / 10
    print(f"  P&L at {q:.0%} pct of top wallets: {top_pnl.quantile(q):,.2f} USDC")

P&L threshold (95th pct, training): 226.79 USDC
Top-5% wallet count:                45,407
  P&L at 0% pct of top wallets: 226.80 USDC
  P&L at 10% pct of top wallets: 284.16 USDC
  P&L at 20% pct of top wallets: 350.95 USDC
  P&L at 30% pct of top wallets: 456.75 USDC
  P&L at 40% pct of top wallets: 604.02 USDC
  P&L at 50% pct of top wallets: 797.23 USDC
  P&L at 60% pct of top wallets: 1,113.39 USDC
  P&L at 70% pct of top wallets: 1,613.03 USDC
  P&L at 80% pct of top wallets: 2,490.20 USDC
  P&L at 90% pct of top wallets: 5,195.76 USDC
  P&L at 100% pct of top wallets: 7,357,349.18 USDC


In [48]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "token_id", "outcome",
    "quantity", "price", "usdc_amount", "position",
    "final_value_usdc", "trade_pnl",
    "token_winner", "final_price",
    "tx_hash",
    "is_train",
]

EXPORT_READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "condition_id", "token_id",
    "outcome", "price", "quantity", "usdc_amount", "position", "wallet", "side",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

total_top_rows = 0
total_top_train_rows = 0
top_trades_preview = None
output_files = []

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=EXPORT_READ_COLS)
    enriched_chunk = enrich_trade_chunk(raw_chunk)
    if len(enriched_chunk) == 0:
        continue

    top_chunk = enriched_chunk[enriched_chunk["wallet"].isin(top_wallets)].copy()
    if len(top_chunk) == 0:
        continue

    is_buy_fill = top_chunk["side"] == "BUY"
    top_chunk["trade_pnl"] = (
        (top_chunk["final_value_usdc"] - top_chunk["usdc_amount"]).where(
            is_buy_fill,
            top_chunk["usdc_amount"] - top_chunk["final_value_usdc"],
        )
    )
    top_chunk["is_train"] = top_chunk["dt"].dt.date <= END_DATE_TRAIN

    export_chunk = top_chunk[EXPORT_COLS]

    out_file = OUT_DIR / f.name
    export_chunk.to_parquet(out_file, index=False)
    output_files.append(out_file)

    if top_trades_preview is None:
        top_trades_preview = export_chunk.head(5).copy()

    total_top_rows += len(export_chunk)
    total_top_train_rows += int(export_chunk["is_train"].sum())

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed export shard {i}/{len(trade_files)} | "
            f"accumulated rows: {total_top_rows:,} | output shards: {len(output_files):,}"
        )

if not output_files:
    empty_file = OUT_DIR / "part-00000.parquet"
    pd.DataFrame(columns=EXPORT_COLS).to_parquet(empty_file, index=False)
    output_files = [empty_file]

top_trades_count = total_top_rows

print(f"Total expanded rows for top wallets: {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
print(f"\nSaved partitioned output → {OUT_DIR}")


Processed export shard 16/16 | accumulated rows: 39,289,062 | output shards: 16
Total expanded rows for top wallets: 39,289,062
  training rows: 27,380,952
  test rows:     11,908,110
Output shards:  16

Saved partitioned output → ../data/polygon_trades_processed


In [49]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview


In [50]:
# WALLET = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

# wallet_parts = []
# for f in trade_files:
#     part = pd.read_parquet(
#         f,
#         columns=[
#             "tx_hash", "log_index", "block_number", "block_timestamp", "trade_date",
#             "condition_id", "token_id", "outcome", "wallet", "side",
#             "price", "quantity", "usdc_amount", "position",
#         ],
#     )
#     part = part[part["wallet"] == WALLET]
#     if len(part) == 0:
#         continue
#     part = part.copy()
#     part["dt"] = pd.to_datetime(part["block_timestamp"], unit="s", utc=True)
#     wallet_parts.append(part)

# wallet_rows = pd.concat(wallet_parts, ignore_index=True) if wallet_parts else pd.DataFrame()
# wallet_rows


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.
It also reports whether the exchange appears in the currently filtered dataset
(appearance depends on the chosen date range).


In [51]:
# ── Unit test: CTF Exchange contract is not selected as a top wallet ──────────
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

# Presence in grouped depends on date filtering; this is informational.
exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

# Must always hold regardless of date range.
assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")


INFO  CTF Exchange appears in grouped:  5 rows
PASS  CTF Exchange not in top_wallets: top_wallets has 45,407 entries


In [52]:
WALLET = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"
if len(wallet_rows) == 0:
    print(f"No rows found for wallet {WALLET}")
    wallet_rows
else:
    wallet_rows.sort_values("dt").tail(20)
